In [8]:
%pip install -qU --pre langchain langchain-text-splitters langchain-community langchain-core langgraph
%pip install -qU --pre "langchain[openai]"
%pip install -qU --pre "langchain-openai==1.0.0a4"
%pip install -qU --pre --ignore-installed langchain-chroma

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("LangSmith API key: ")

In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

In [4]:
import getpass
import os

from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-5-mini",)

In [5]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [9]:
# from langchain_core.vectorstores import InMemoryVectorStore
# vector_store = InMemoryVectorStore(embeddings)
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [18]:
from itertools import chain
ids = list(chain(range(1,8801), range(10001, 10055), range(270001,270396)))
print(len(ids))

9249


In [10]:
# Load Docs and Insert them into Vector Store
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# general conference 1 - 8800
# journal of discourses 10001 - 10054
# joseph smith teachings 270001 - 270395
from itertools import chain
ids = list(chain(range(1,8801), range(10001, 10055), range(270001,270396)))
BATCH_SIZE=1000

web_paths = [f"https://scriptures.byu.edu/content/talks_ajax/{number}" for number in range(1,8801)]


for i in range(0, len(web_paths), BATCH_SIZE)
    # Load and chunk contents of the blog
    loader = WebBaseLoader(
        web_paths=web_paths,
        bs_kwargs=dict(
            parse_only=bs4.SoupStrainer(
                id=("talkcontent")
            )
        ),
    )
    docs = loader.load()

    #print(docs[0].page_content[:1000])
    
    # Split Docs into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)
    all_splits = text_splitter.split_documents(docs)
    
    # Index chunks
    # WARNING This will use many api calls
    document_ids = vector_store.add_documents(documents=all_splits)
    print("done")

USER_AGENT environment variable not set, consider setting it to identify your requests.


InternalError: ValueError: Batch size of 96931 is greater than max batch size of 5461

In [ ]:
# NON-AGENT RAG setup
from langchain.agents import create_agent, AgentState
from langchain.messages import MessageLikeRepresentation


def prompt_with_context(state: AgentState) -> list[MessageLikeRepresentation]:
    """Inject context into state messages."""
    last_query = state["messages"][-1].text
    retrieved_docs = vector_store.similarity_search(last_query)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful assistant. Use the following context in your response:"
        f"\n\n{docs_content}"
    )

    return [{"role": "system", "content": system_message}, *list(state["messages"])]


agent = create_agent(llm, tools=[], system_prompt=prompt_with_context)

# Invoke query
query = "What can I do to help alleviate suffering?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

# Construct a tool for retrieving context
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from many talks and articles given by Prophets."
    "Use the tool to help answer user queries as if you were a Prophet.  Quote directly from the Prophet or from the Scriptures if you can."
)
agent = create_agent(llm, tools=[retrieve_context], system_prompt=prompt)

In [ ]:
# Invoke query
query = "What can I improve my relationship with my wife?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()